# 08 — vLLM: RULER Benchmark

This notebook runs the same RULER benchmark as notebook 07, but using
[vLLM](https://github.com/vllm-project/vllm) with Qwen3-8B and
KV cache compression via vLLM's built-in compression support.

We test two compression algorithms at multiple ratios:
- **full_replacement** — full KV cache replacement after prefill
- **filtering** — token filtering during decoding

Scoring uses `calculate_metrics` from the kvpress evaluation framework
(same scoring as the [kvpress leaderboard](https://huggingface.co/spaces/nvidia/kvpress-leaderboard))
for apples-to-apples comparison with notebook 07.

Results are saved to `results/vllm_ruler/` for comparison in later notebooks.

## Configuration

In [ ]:
MODEL_NAME = "Qwen/Qwen3-8B"

COMPRESSION_RATIOS = [0.01, 0.25, 0.50, 0.75]

RULER_DATA_DIRS = ["4096", "8192"]

FRACTION = 0.01

SEED = 42

MAX_NEW_TOKENS = 128

PRESS_CONFIGS = {
    "full_replacement": lambda cr: {
        "model": MODEL_NAME,
        "dtype": "auto",
        "gpu_memory_utilization": 0.90,
        "trust_remote_code": True,
        "attention_config": {"backend": "FLASH_ATTN"},
        "kv_compression_algorithm": "full_replacement",
        "kv_compression_ratio": cr,
        "enable_prefix_caching": False,
    },
    "filtering": lambda cr: {
        "model": MODEL_NAME,
        "dtype": "auto",
        "gpu_memory_utilization": 0.90,
        "trust_remote_code": True,
        "attention_config": {"backend": "FLASH_ATTN"},
        "kv_compression_algorithm": "filtering",
        "kv_compression_ratio": cr,
    },
}

In [ ]:
import sys
import builtins

_original_print = builtins.print

def print(*args, **kwargs):
    _original_print(*args, **kwargs)
    if sys.stdout is not sys.__stdout__:
        kwargs['file'] = sys.__stdout__
        kwargs['flush'] = True
        _original_print(*args, **kwargs)

In [ ]:
import sys
import os

FORK_DIR = "/opt/app-root/src/vllm-fork"

if os.path.isdir(FORK_DIR) and os.listdir(FORK_DIR):
    sys.path.insert(0, FORK_DIR)
    import vllm
    print(f"Using FORK vLLM (version: {vllm.__version__})")
else:
    import vllm
    print(f"Using SYSTEM vLLM (version: {vllm.__version__})")

In [ ]:
import gc
import torch

if not torch.cuda.is_available():
    raise RuntimeError("No CUDA GPU detected.")

vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
allocated_gb = torch.cuda.memory_allocated() / 1e9
reserved_gb = torch.cuda.memory_reserved() / 1e9

print(f"GPU:        {torch.cuda.get_device_name(0)}")
print(f"VRAM:       {vram_gb:.1f} GB total")
print(f"Allocated:  {allocated_gb:.2f} GB")
print(f"Reserved:   {reserved_gb:.2f} GB")
print(f"Free:       {vram_gb - reserved_gb:.1f} GB (approx)")

if allocated_gb > 1.0:
    print(
        "\n\u26a0  GPU memory is not free \u2014 a model from another notebook may still be loaded.\n"
        "   Restart this kernel before proceeding."
    )

def cleanup_vllm(llm):
    llm.llm_engine.engine_core.shutdown()
    del llm
    gc.collect()
    torch.cuda.empty_cache()

def get_gpu_memory_used_gb() -> float:
    """Actual GPU memory used, measured at the CUDA driver level."""
    free, total = torch.cuda.mem_get_info()
    return (total - free) / 1e9

In [ ]:
KVPRESS_FORK_DIR = "/opt/app-root/src/kvpress-fork"
EVAL_DIR = os.path.join(KVPRESS_FORK_DIR, "evaluation")
sys.path.insert(0, EVAL_DIR)
from benchmarks.ruler.calculate_metrics import calculate_metrics as ruler_calculate_metrics
print(f"RULER scoring from: {EVAL_DIR}")

## 1. Load RULER Dataset

In [ ]:
from datasets import load_dataset

ruler_datasets = {}
for data_dir in RULER_DATA_DIRS:
    df = load_dataset("simonjegou/ruler", data_dir=data_dir, split="test").to_pandas()
    if FRACTION < 1.0:
        df = df.sample(frac=FRACTION, random_state=SEED)
    df["context_length"] = int(data_dir)
    ruler_datasets[data_dir] = df
    tasks = sorted(df["task"].unique())
    print(f"RULER {data_dir}: {len(df)} examples, {len(tasks)} tasks")
    print(f"  Tasks: {tasks}")

## 2. Prepare Prompts

Apply the model's chat template to each RULER example to produce
the final prompt for vLLM batch inference.

In [ ]:
import pandas as pd
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)

ruler_df = pd.concat(ruler_datasets.values(), ignore_index=True)

prompts = []
for _, row in ruler_df.iterrows():
    user_msg = row["context"] + "\n\n" + row["question"]
    messages = [{"role": "user", "content": user_msg}]
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True,
    )
    if row["answer_prefix"]:
        prompt += row["answer_prefix"]
    prompts.append(prompt)

ruler_df["prompt"] = prompts
print(f"Prepared {len(ruler_df)} prompts")

## 3. Run Batch Inference

For each (algorithm, compression_ratio) combination, create a vLLM engine
and process all prompts in batch. The engine is destroyed between configs
to free GPU memory.

In [ ]:
import time
import random
import numpy as np
from vllm import LLM, SamplingParams

# Deterministic seeds — matches evaluate.py _setup_deterministic_seeds()
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)
torch.cuda.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

sampling_params = SamplingParams(
    temperature=0.0,
    max_tokens=MAX_NEW_TOKENS,
)

prompts = ruler_df["prompt"].tolist()

all_metrics = {}
summary_rows = []
all_predictions = []

configs = [("no_press", 0.0, None)]
for press_name, press_factory in PRESS_CONFIGS.items():
    for ratio in COMPRESSION_RATIOS:
        configs.append((press_name, ratio, press_factory(ratio)))

for press_name, ratio, press_kwargs in configs:
    llm = None
    try:
        label = f"{press_name} | ratio={ratio}"
        print(f"\n{'='*60}")
        print(f"Running: {label} ({len(prompts)} prompts)")
        print(f"{'='*60}")

        if press_kwargs is not None:
            llm = LLM(**press_kwargs)
        else:
            llm = LLM(
                model=MODEL_NAME,
                dtype="auto",
                gpu_memory_utilization=0.90,
                trust_remote_code=True,
                attention_config={"backend": "FLASH_ATTN"},
                enable_prefix_caching=False,
            )

        mem_before = get_gpu_memory_used_gb()
        start = time.perf_counter()

        outputs = llm.generate(prompts, sampling_params)

        batch_elapsed = time.perf_counter() - start
        mem_after = get_gpu_memory_used_gb()
        peak_mem = max(mem_before, mem_after)

        # Build scoring DataFrame with original answer column types from .to_pandas()
        df_eval = ruler_df[["task", "answer", "context_length"]].copy()
        df_eval["predicted_answer"] = [o.outputs[0].text.strip() for o in outputs]

        # Score per context_length — same flow as evaluate.py
        # .copy() ensures calculate_metrics receives an owned DataFrame,
        # matching evaluate.py which passes self.df directly
        for ctx_len, df_ctx in df_eval.groupby("context_length"):
            metrics = ruler_calculate_metrics(df_ctx.copy())
            key = f"{press_name}__{ratio}__{ctx_len}"
            all_metrics[key] = metrics

            avg_score = sum(m["string_match"] for m in metrics.values()) / len(metrics)
            summary_rows.append({
                "press": press_name, "compression_ratio": ratio,
                "context_length": ctx_len, "avg_score": round(avg_score, 2),
                "mean_time": round(batch_elapsed / len(prompts), 3),
                "peak_gpu_mem_gb": round(peak_mem, 3),
            })

        # Collect predictions for saving
        df_preds = df_eval.copy()
        df_preds["framework"] = "vllm"
        df_preds["press"] = press_name
        df_preds["compression_ratio"] = ratio
        df_preds["elapsed_sec"] = round(batch_elapsed / len(prompts), 3)
        df_preds["peak_gpu_mem_gb"] = round(peak_mem, 3)
        all_predictions.append(df_preds)

        total_gen_tokens = sum(len(o.outputs[0].token_ids) for o in outputs)
        throughput = total_gen_tokens / batch_elapsed if batch_elapsed > 0 else 0

        print(f"  Batch done in {batch_elapsed:.1f}s — {throughput:.1f} tok/s — peak mem={peak_mem:.2f} GB")
    finally:
        if llm is not None:
            cleanup_vllm(llm)

print(f"\nTotal configurations: {len(summary_rows)}")

## 4. Results

Scores computed using `calculate_metrics` from the kvpress evaluation
framework — same string-match scoring as the kvpress leaderboard.

In [ ]:
summary = pd.DataFrame(summary_rows)
print(summary.to_string(index=False))

In [ ]:
for key, metrics in sorted(all_metrics.items()):
    print(f"\n{key}")
    for task, scores in sorted(metrics.items()):
        print(f"  {task:30s}: {scores['string_match']:.2f}")

## 5. Save Results

In [ ]:
import json

os.makedirs("results/vllm_ruler", exist_ok=True)

predictions_path = "results/vllm_ruler/predictions.csv"
df_all = pd.concat(all_predictions, ignore_index=True)
df_all.to_csv(predictions_path, index=False)
print(f"Saved predictions to {predictions_path}")

metrics_path = "results/vllm_ruler/metrics.json"
with open(metrics_path, "w") as f:
    json.dump(all_metrics, f, indent=2)
print(f"Saved metrics to {metrics_path}")